# NGBoost

In [1]:
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

model_path = Path("model") / "NGBoost_6_air.pkl"
in_path    = Path("data") / "new_with_O2_CO2_500.xlsx"
out_path   = Path("predict") / "predict_air_ngboost_6_500.csv"

model = joblib.load(model_path)

df = pd.read_excel(in_path)

name_col = None
for cand in ["compounds", "compound", "Compound", "name", "material"]:
    if cand in df.columns:
        name_col = cand
        break

feature_names = ["CurrentDensity", "O2", "CO2"]
missing = [c for c in feature_names if c not in df.columns]

X_new = df[feature_names].to_numpy(dtype=np.float32)

pred_mean = np.asarray(model.predict(X_new), dtype=float).ravel()

if hasattr(model, "pred_dist"):
    dist = model.pred_dist(X_new)
else:
    try:
        dist = model.named_steps["ngboost"].pred_dist(X_new)
    except Exception as e:
        raise RuntimeError("no pred_dist") from e

try:
    pred_std = np.asarray(dist.scale, dtype=float).ravel()
except Exception:
    q16 = np.asarray(dist.ppf(0.158655), dtype=float).ravel()
    q84 = np.asarray(dist.ppf(0.841345), dtype=float).ravel()
    pred_std = (q84 - q16) / 2.0

out = pd.DataFrame({
    "compounds": df[name_col].astype(str).values,
    "pred_mean": pred_mean,
    "pred_std":  pred_std
})

out_path.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"saved: {out_path}")


saved: predict\predict_air_ngboost_6_500.csv


In [7]:
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

model_path = Path("model") / "NGBoost_1_air.pkl"
in_path    = Path("data") / "new_with_O2_CO2_50.xlsx"
out_path   = Path("predict") / "predict_air_ngboost_1_50.csv"

model = joblib.load(model_path)

df = pd.read_excel(in_path)

name_col = None
for cand in ["compounds", "compound", "Compound", "name", "material"]:
    if cand in df.columns:
        name_col = cand
        break
if name_col is None:
    raise ValueError("找不到样本名列（如 'compounds' 或 'compound'）。请在输入表中包含其中之一。")

feature_names = ["CurrentDensity", "O2", "CO2"]
missing = [c for c in feature_names if c not in df.columns]
if missing:
    raise ValueError(f"输入数据缺少列: {missing}")

X_new = df[feature_names].to_numpy(dtype=np.float32)

pred_mean = np.asarray(model.predict(X_new), dtype=float).ravel()

if hasattr(model, "pred_dist"):
    dist = model.pred_dist(X_new)
else:
    try:
        dist = model.named_steps["ngboost"].pred_dist(X_new)
    except Exception as e:
        raise RuntimeError(" pred_dist error") from e

try:
    pred_std = np.asarray(dist.scale, dtype=float).ravel()
except Exception:
    q16 = np.asarray(dist.ppf(0.158655), dtype=float).ravel()
    q84 = np.asarray(dist.ppf(0.841345), dtype=float).ravel()
    pred_std = (q84 - q16) / 2.0

out = pd.DataFrame({
    "compounds": df[name_col].astype(str).values,
    "pred_mean": pred_mean,
    "pred_std":  pred_std
})

out_path.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"saved: {out_path}")


saved: predict\predict_air_ngboost_1_50.csv
